In [1]:
import re
import time
import random
import requests
import pandas as pd

from pathlib import Path
from bs4 import BeautifulSoup

ROOT = Path.cwd().parent
print(ROOT)

c:\Users\sebas\PycharmProjects\Git\BoxOffice_Oracle


In [2]:
DATA_PROCESSED = ROOT/"data/processed"
CACHE_DIR = ROOT/"data/cache/the_numbers_pages"

INPUT_PATH = DATA_PROCESSED / "scrape_ready_us_production_filtered.csv"
OUTPUT_PATH = DATA_PROCESSED / "the_numbers_plot_franchise.csv"

scrape_df = pd.read_csv(INPUT_PATH)

scrape_df = scrape_df[
    scrape_df["the_numbers_url"].notna()
].drop_duplicates(subset="tconst").reset_index(drop=True)

print(scrape_df.shape)
scrape_df.head()

(2316, 9)


,tconst,primaryTitle,startYear,the_numbers_url,found_url,normalized_title,kaggle_year,kaggle_title,Production_Countries
0,tt0120667,Fantastic Four,2005.0,https://www.the-numbers.com/movie/Fantastic-Fo...,True,fantastic four,2005,Fantastic Four,"Germany, United States of America"
1,tt0121164,Corpse Bride,2005.0,https://www.the-numbers.com/movie/Corpse-Bride,True,corpse bride,2005,Corpse Bride,"United Kingdom, United States of America"
2,tt0121766,Star Wars: Episode III - Revenge of the Sith,2005.0,https://www.the-numbers.com/movie/Star-Wars-Ep...,True,star wars episode iii revenge of the sith,2005,Star Wars: Episode III - Revenge of the Sith,United States of America
3,tt0167190,Hellboy,2004.0,https://www.the-numbers.com/movie/Hellboy,True,hellboy,2004,Hellboy,"Czech Republic, United States of America"
4,tt0200465,The Bank Job,2008.0,https://www.the-numbers.com/movie/Bank-Job-The,True,the bank job,2008,The Bank Job,"Australia, United Kingdom, United States of Am..."


In [3]:
headers = {
    "User-Agent": "Mozilla/5.0"
}

def clean_cache_name(text):
    text = str(text)
    text = re.sub(r"[^a-zA-Z0-9]+", "_", text)
    return text.strip("_")[:120]


def fetch_the_numbers_html(row):
    tconst = row["tconst"]
    url = row["the_numbers_url"]

    cache_path = CACHE_DIR / f"{clean_cache_name(tconst)}.html"

    if cache_path.exists():
        return cache_path.read_text(encoding="utf-8", errors="ignore")

    response = requests.get(url, headers=headers, timeout=25)
    response.raise_for_status()

    html = response.text
    cache_path.write_text(html, encoding="utf-8", errors="ignore")

    time.sleep(2.5 + random.uniform(0, 1))

    return html

In [4]:
def get_lines_from_html(html):
    soup = BeautifulSoup(html, "html.parser")

    lines = [
        re.sub(r"\s+", " ", x).strip()
        for x in soup.stripped_strings
    ]

    return [line for line in lines if line]

In [18]:
def get_field(lines, label):

    normalized_label = label.rstrip(":")

    for i, line in enumerate(lines):

        clean_line = line.strip()

        if clean_line.rstrip(":") == normalized_label:

            # look ahead until non-empty
            for j in range(i + 1, len(lines)):

                candidate = lines[j].strip()

                if candidate:
                    return candidate

        if clean_line.startswith(f"{normalized_label}:"):

            value = clean_line.split(":", 1)[1].strip()

            if value:
                return value

    return None

In [19]:
def extract_synopsis(lines):

    synopsis = get_field(lines, "Synopsis")

    if synopsis:
        synopsis = synopsis.strip()

    return synopsis

In [20]:
def extract_franchise(lines):

    franchise = get_field(lines, "Franchise")

    if franchise:
        franchise = franchise.strip()

    return franchise

In [21]:
def parse_plot_and_franchise(html):

    lines = get_lines_from_html(html)

    return {
        "plot_summary": extract_synopsis(lines),
        "franchise": extract_franchise(lines),
    }

In [22]:
test_rows = scrape_df.sample(10, random_state=42)

test_results = []

for _, row in test_rows.iterrows():
    html = fetch_the_numbers_html(row)
    parsed = parse_plot_and_franchise(html)

    test_results.append({
        "tconst": row["tconst"],
        "primaryTitle": row["primaryTitle"],
        "the_numbers_url": row["the_numbers_url"],
        **parsed
    })

test_df = pd.DataFrame(test_results)

test_df[[
    "primaryTitle",
    "plot_summary",
    "franchise"
]]

,primaryTitle,plot_summary,franchise
0,Breakthrough,The true story of one mother’s unfaltering lov...,NaN
1,The Covenant,NaN,NaN
2,Miracle,NaN,NaN
3,Bones and All,"A story of first love between Maren, a young w...",NaN
4,Seed of Chucky,NaN,Child's Play
5,She's the Man,Viola Johnson is in a real jam. Complications ...,NaN
6,The Menu,"Set in the world of high-end culinary culture,...",NaN
7,The Ministry of Ungentlemanly Warfare,The story of the first-ever special forces org...,NaN
8,Spanglish,NaN,NaN
9,Transformers: Age of Extinction,"An epic battle left a great city torn, but wit...",Transformers


In [23]:
rows = []

start_time = time.time()

for i, (_, row) in enumerate(scrape_df.iterrows(), start=1):

    record = {
        "tconst": row["tconst"],
        "primaryTitle": row.get("primaryTitle"),
        "startYear": row.get("startYear"),
        "the_numbers_url": row["the_numbers_url"],
        "scrape_success": False,
        "scrape_error": None,
    }

    try:
        html = fetch_the_numbers_html(row)
        parsed = parse_plot_and_franchise(html)

        record.update(parsed)
        record["scrape_success"] = True

    except Exception as e:
        record["scrape_error"] = str(e)

    rows.append(record)

    if i % 100 == 0:
        elapsed = (time.time() - start_time) / 60
        found_plot = sum(r.get("plot_summary") is not None for r in rows)
        found_franchise = sum(r.get("franchise") is not None for r in rows)

        print(
            f"[{i}/{len(scrape_df)}] "
            f"Plot found: {found_plot} | "
            f"Franchise found: {found_franchise} | "
            f"Elapsed: {elapsed:.2f} min"
        )

plot_franchise_df = pd.DataFrame(rows)

plot_franchise_df.to_csv(OUTPUT_PATH, index=False)

print(plot_franchise_df.shape)
plot_franchise_df.head()

[100/2316] Plot found: 29 | Franchise found: 34 | Elapsed: 0.15 min
[200/2316] Plot found: 56 | Franchise found: 70 | Elapsed: 0.29 min
[300/2316] Plot found: 89 | Franchise found: 98 | Elapsed: 0.42 min
[400/2316] Plot found: 122 | Franchise found: 122 | Elapsed: 0.57 min
[500/2316] Plot found: 154 | Franchise found: 153 | Elapsed: 0.74 min
[600/2316] Plot found: 190 | Franchise found: 183 | Elapsed: 0.93 min
[700/2316] Plot found: 241 | Franchise found: 209 | Elapsed: 1.11 min
[800/2316] Plot found: 287 | Franchise found: 236 | Elapsed: 1.30 min
[900/2316] Plot found: 347 | Franchise found: 266 | Elapsed: 1.50 min
[1000/2316] Plot found: 417 | Franchise found: 295 | Elapsed: 1.69 min
[1100/2316] Plot found: 491 | Franchise found: 332 | Elapsed: 1.89 min
[1200/2316] Plot found: 577 | Franchise found: 369 | Elapsed: 2.09 min
[1300/2316] Plot found: 670 | Franchise found: 399 | Elapsed: 2.31 min
[1400/2316] Plot found: 766 | Franchise found: 434 | Elapsed: 2.50 min
[1500/2316] Plot foun

,tconst,primaryTitle,startYear,the_numbers_url,scrape_success,scrape_error,plot_summary,franchise
0,tt0120667,Fantastic Four,2005.0,https://www.the-numbers.com/movie/Fantastic-Fo...,True,None,Scientist Reed Richards persuades his arrogant...,Fantastic Four
1,tt0121164,Corpse Bride,2005.0,https://www.the-numbers.com/movie/Corpse-Bride,True,None,NaN,NaN
2,tt0121766,Star Wars: Episode III - Revenge of the Sith,2005.0,https://www.the-numbers.com/movie/Star-Wars-Ep...,True,None,"Years after the onset of the Clone Wars, the n...",Star Wars
3,tt0167190,Hellboy,2004.0,https://www.the-numbers.com/movie/Hellboy,True,None,NaN,Hellboy
4,tt0200465,The Bank Job,2008.0,https://www.the-numbers.com/movie/Bank-Job-The,True,None,Self-reformed petty criminal Terry Leather has...,NaN


In [24]:
plot_franchise_df[[
    "plot_summary",
    "franchise",
    "scrape_success",
    "scrape_error"
]].isna().mean()

plot_summary      0.291451
franchise         0.667530
scrape_success    0.000000
scrape_error      1.000000
dtype: float64

In [25]:
plot_franchise_df["has_plot_summary"] = (
    plot_franchise_df["plot_summary"].notna()
)

In [26]:
plot_franchise_df[
    plot_franchise_df["plot_summary"].isna()
][["primaryTitle", "the_numbers_url"]].sample(20, random_state=42)

,primaryTitle,the_numbers_url
580,The Strangers,https://www.the-numbers.com/movie/Strangers-Th...
436,Hoodwinked,https://www.the-numbers.com/movie/Hoodwinked
2182,Truth or Dare,https://www.the-numbers.com/movie/Truth-or-Dar...
870,The Proposal,https://www.the-numbers.com/movie/Proposal-The
546,State of Play,https://www.the-numbers.com/movie/State-of-Play
77,Cellular,https://www.the-numbers.com/movie/Cellular
1577,No Good Deed,https://www.the-numbers.com/movie/No-Good-Deed
371,Surf's Up,https://www.the-numbers.com/movie/Surfs-Up-(2007)
1954,Bride Wars,https://www.the-numbers.com/movie/Bride-Wars
734,The Forbidden Kingdom,https://www.the-numbers.com/movie/Forbidden-Ki...


In [27]:
plot_franchise_df.to_csv(
    ROOT/"data/processed/the_numbers_plot_franchise.csv",
    index=False
)

print("Saved successfully.")

Saved successfully.


In [28]:
DATA_RAW = ROOT/"data/raw"
model_df = pd.read_csv(
    DATA_RAW/"default_model_df.csv"
)

print(model_df.shape)
model_df.head()

(2255, 47)


,Unnamed: 0,tconst,primaryTitle,startYear,the_numbers_url,scrape_success,scrape_error,opening_weekend_gross,opening_theaters,domestic_release_date,...,actor_2,actor_3,actor_4,actor_5,actor_6,director_name,writer_name,actor_1_name,actor_2_name,actor_3_name
0,0,tt0120667,Fantastic Four,2005.0,https://www.the-numbers.com/movie/Fantastic-Fo...,True,NaN,56061504.0,3602.0,2005-07-08,...,nm0004821,nm0262635,nm0004695,nm0573037,nm0512934,Tim Story,Mark Frost,Ioan Gruffudd,Michael Chiklis,Chris Evans
1,1,tt0121164,Corpse Bride,2005.0,https://www.the-numbers.com/movie/Corpse-Bride,True,NaN,19145480.0,3204.0,2005-09-16,...,nm0000307,nm0001833,nm0001808,nm0925768,NaN,Tim Burton,John August,Johnny Depp,Helena Bonham Carter,Emily Watson
2,2,tt0121766,Star Wars: Episode III - Revenge of the Sith,2005.0,https://www.the-numbers.com/movie/Star-Wars-Ep...,True,NaN,108435841.0,3661.0,2005-05-19,...,nm0000204,nm0000191,nm0000168,nm0001519,nm0001751,George Lucas,George Lucas,Hayden Christensen,Natalie Portman,Ewan McGregor
3,3,tt0167190,Hellboy,2004.0,https://www.the-numbers.com/movie/Hellboy,True,NaN,23172440.0,3028.0,2004-04-02,...,nm0427964,nm0004757,nm0000457,nm1140344,nm0734558,Guillermo del Toro,Guillermo del Toro,Ron Perlman,Doug Jones,Selma Blair
4,4,tt0200465,The Bank Job,2008.0,https://www.the-numbers.com/movie/Bank-Job-The,True,NaN,5935256.0,1603.0,2008-03-07,...,nm0004787,nm1304386,nm0990547,nm0269077,nm0202810,Roger Donaldson,Dick Clement,Jason Statham,Saffron Burrows,Stephen Campbell Moore


In [29]:
model_df = model_df.merge(
    plot_franchise_df[[
        "tconst",
        "plot_summary",
        "has_plot_summary",
        "franchise"
    ]],
    on="tconst",
    how="left"
)

In [30]:
model_df.to_csv(
    DATA_RAW/"default_model_w_franchise_n_plot.csv",
    index=False
)